# ARC-3 RFT round 1 — LoRA fine-tune Qwen3.6-27B on the base model's success trajectories

In [ ]:

import subprocess, sys
# Qwen3.6 (arch 'qwen3_5') needs transformers-MAIN. Install verbosely + check=True so failures surface.
r=subprocess.run([sys.executable,"-m","pip","install","-U","--force-reinstall","--no-deps",
                  "git+https://github.com/huggingface/transformers.git"], capture_output=True, text=True)
print("pip rc:", r.returncode); print(r.stdout[-800:]); print("ERR:", r.stderr[-800:])
subprocess.run([sys.executable,"-m","pip","install","-q","-U","peft","trl","datasets","accelerate"], check=False)
import importlib, transformers; importlib.reload(transformers)
print("transformers", transformers.__version__, "|", transformers.__file__)
# fast arch check: load ONLY the config (no 54GB download)
from transformers import AutoConfig
try:
    cfg=AutoConfig.from_pretrained("Qwen/Qwen3.6-27B", trust_remote_code=True)
    print("CONFIG OK -> model_type:", cfg.model_type, "| TRANSFORMERS SUPPORTS IT")
except Exception as e:
    print("CONFIG FAILED:", str(e)[:200])


In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = "Qwen/Qwen3.6-27B"   # BF16 base from HF (internet on). LoRA fits 96GB without quantization.
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map={"":0}, trust_remote_code=True)
model.config.use_cache = False
print("model loaded:", model.config.model_type, sum(p.numel() for p in model.parameters())/1e9, "B params")


In [ ]:

from datasets import load_dataset
import glob
path = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]
ds = load_dataset("json", data_files=path, split="train")
print("RFT examples:", len(ds), "| example keys:", ds[0].keys())
# quick token-length sanity
lens = [len(tok.apply_chat_template(ds[i]["messages"], tokenize=True)) for i in range(min(50,len(ds)))]
print(f"seq len (first 50): median {sorted(lens)[len(lens)//2]}, max {max(lens)}")


In [ ]:

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                         target_modules="all-linear", task_type="CAUSAL_LM")
args = SFTConfig(
    output_dir="/kaggle/working/rft_adapter",
    num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
    max_length=3072, packing=False, bf16=True, gradient_checkpointing=True,
    logging_steps=10, save_strategy="epoch", report_to="none",
    assistant_only_loss=True,   # loss on assistant tokens only (needs a chat template with {% generation %})
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=peft_config, processing_class=tok)
trainer.train()
trainer.save_model("/kaggle/working/rft_adapter")
tok.save_pretrained("/kaggle/working/rft_adapter")
print("=== adapter saved to /kaggle/working/rft_adapter ===")
import os; print(os.listdir("/kaggle/working/rft_adapter"))


In [ ]:

# Kaggle expects an output; the adapter dir is the real product.
import pandas as pd
pd.DataFrame([["1_0","1",True,0]],columns=["row_id","game_id","end_of_game","score"]).to_parquet("/kaggle/working/submission.parquet",index=False)
print("done")
